In [ ]:
%pip install ../requirements.txt

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.7 MB ? eta -:--:--
   ------------ --------------------------- 0.5/1.7 MB 985.5 kB/s eta 0:00:02
   ------------------ --------------------- 0.8/1.7 MB 1.1 MB/s eta 0:00:01
   ------------------------------ --------- 1.3/1.7 MB 1.4 MB/s eta 0:00:01
   ---------------------------------------- 1.7/1.7 MB 1.6 MB/s  0:00:01
   ---------------------------------------- 0.0/350.8 MB ? eta -:--:--
   ---------------------------------------- 0.5/350.8 MB 2.8 MB/s eta 0:02:06
   ---------------------------------------- 1.0/350.8 MB 3.0 MB/s eta 0:01:59
   ---------------------------------------- 2.1/350.8 MB 3.6 MB/s eta 0:01:39
   -----------------------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
googleapis-common-protos 1.69.2 requires protobuf!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.34.1 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.34.1 which is incompatible.
google-api-core 2.24.2 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.19.5, but you have protobuf 7.34.1 which is incompatible.
grpcio-status 1.71.0 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.34.1 which is incompatible.
proto-plus 1.26.1 requires protobuf<7.0.0,>=3.19.0, but you have protobuf 7.34.1 which is incompatible.
tensorflow-intel 2.18.0 requires ml-dtypes<0.5.0,>=0.4.0, but you have ml-dtypes 0.5.4 whi

In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader("../data/AITrainingDocument.pdf")
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = text_splitter.split_documents(docs)

print(len(chunks))
print(chunks[0].page_content)


273
User Agreement 
1. Introduction 
This User Agreement, the Mobile Application Terms of Use, and all policies and additional terms 
posted on and in our sites, applications, tools, and services (collectively "Services") set out the terms


In [2]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_db = FAISS.from_documents(chunks, embeddings)

vector_db.save_local("../vectordb")

W0402 00:29:45.727000 14580 torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [4]:
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate

llm = OllamaLLM(model="mistral", num_ctx=2048)

template = """Context:
{context}

Question: {question}
Answer strictly based on the provided context:"""

prompt = PromptTemplate(template=template, input_variables=["context", "question"])

pull mistral model manually:

In [ ]:
%ollama pull mistral

In [5]:
query = "What are the main obligations mentioned in the text?"

retriever = vector_db.as_retriever(search_kwargs={"k": 3})
retrieved_docs = retriever.invoke(query)

context = "\n\n".join([doc.page_content for doc in retrieved_docs])
formatted_prompt = prompt.format(context=context, question=query)

for chunk in llm.stream(formatted_prompt):
    print(chunk, end="", flush=True)

 The main obligations mentioned in the text include:

1. Ownership and control of all content provided by the user (represented by "you") for use in connection with the Services. This also implies that the user should have all necessary agreements in place for any such content.

2. Compliance with the terms and conditions outlined in this User Agreement, which includes sections on Fees and Taxes, Content, Holds and Restricted Funds, Additional Terms, Payment Services, Disclaimer of Warranties; Limitation of Liability; Release, Indemnity, Legal Disputes, and General.

3. Survival of certain provisions even after termination of the User Agreement, which includes those mentioned in sections Fees and Taxes, Content, Holds and Restricted Funds, Additional Terms, Payment Services, Disclaimer of Warranties; Limitation of Liability; Release, Indemnity, Legal Disputes, and General.

4. The understanding that if any provision of the User Agreement is deemed invalid or unenforceable, it should be

In [6]:
for i, doc in enumerate(retrieved_docs):
    print(f"Source {i+1}:\n{doc.page_content}\n")

Source 1:
connection with our, those assignees', and those sublicensees' use of that content in connection with 
our provision, expansion, and promotion of our Services. 
You represent and warrant that, for all such content you provide, you own or otherwise control all

Source 2:
agreements of the parties. 
The following sections survive any termination of this User Agreement: Fees and Taxes, Content, 
Holds and Restricted Funds, Additional Terms, Payment Services, Disclaimer of Warranties; Limitation 
of Liability; Release, Indemnity, Legal Disputes, and General.

Source 3:
20. General 
Except as otherwise provided in this User Agreement, if any provision of this User Agreement is held 
to be invalid, void or for any reason unenforceable, such provision shall be struck out and shall not

